![dvd_image](dvd_image.jpg)

A DVD rental company needs your help! They want to figure out how many days a customer will rent a DVD for based on some features and has approached you for help. They want you to try out some regression models which will help predict the number of days a customer will rent a DVD for. The company wants a model which yeilds a MSE of 3 or less on a test set. The model you make will help the company become more efficient inventory planning.

The data they provided is in the csv file `rental_info.csv`. It has the following features:
- `"rental_date"`: The date (and time) the customer rents the DVD.
- `"return_date"`: The date (and time) the customer returns the DVD.
- `"amount"`: The amount paid by the customer for renting the DVD.
- `"amount_2"`: The square of `"amount"`.
- `"rental_rate"`: The rate at which the DVD is rented for.
- `"rental_rate_2"`: The square of `"rental_rate"`.
- `"release_year"`: The year the movie being rented was released.
- `"length"`: Lenght of the movie being rented, in minuites.
- `"length_2"`: The square of `"length"`.
- `"replacement_cost"`: The amount it will cost the company to replace the DVD.
- `"special_features"`: Any special features, for example trailers/deleted scenes that the DVD also has.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"`: These columns are dummy variables of the rating of the movie. It takes the value 1 if the move is rated as the column name and 0 otherwise. For your convinience, the reference dummy has already been dropped.

In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
# Import any additional modules and start coding below

In [22]:
# 1. Load data
# ---------------------------------------------------------------------------
df = pd.read_csv("rental_info.csv")
 
# Parse dates
df["rental_date"] = pd.to_datetime(df["rental_date"])
df["return_date"] = pd.to_datetime(df["return_date"])



# 2. Target: rental_length_days
# ---------------------------------------------------------------------------
df["rental_length_days"] = (df["return_date"] - df["rental_date"]).dt.days
 
# ---------------------------------------------------------------------------
# 3. Dummy columns from special_features
# ---------------------------------------------------------------------------
# special_features typically looks like a stringified list/array, e.g.
# '{Trailers,"Deleted Scenes"}'. str.contains handles that fine.
df["deleted_scenes"] = df["special_features"].str.contains(
    "Deleted Scenes", regex=False
).astype(int)
 
df["behind_the_scenes"] = df["special_features"].str.contains(
    "Behind the Scenes", regex=False
).astype(int)



# 4. Build X (features) and y (target)
# ---------------------------------------------------------------------------
# Columns that leak information about the target or aren't usable as
# numeric features directly:
#   - return_date, rental_date -> used to derive the target directly (leak)
#   - rental_length_days       -> the target itself
#   - special_features         -> raw string, already encoded into dummies above
LEAK_COLS = ["rental_length_days", "return_date", "rental_date", "special_features"]
 
X = df.drop(columns=[c for c in LEAK_COLS if c in df.columns])
y = df["rental_length_days"]
 
# Keep only numeric feature columns (defensive, in case any stray text columns remain)
X = X.select_dtypes(include=[np.number])
 
print("Features used in X:", list(X.columns))

# 5. Train/test split
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=9
)




# 6. Feature selection with Lasso (helps trim noisy/collinear features)
# ---------------------------------------------------------------------------
lasso = Lasso(alpha=0.3, random_state=9)
lasso.fit(X_train, y_train)
 
lasso_coef = lasso.coef_
selected_features = X_train.columns[lasso_coef != 0].tolist()
 
# Fall back to all features if Lasso zeroes everything out
if len(selected_features) == 0:
    selected_features = X_train.columns.tolist()
 
print("Lasso-selected features:", selected_features)
 
X_train_lasso = X_train[selected_features]
X_test_lasso = X_test[selected_features]




# 7. Try a couple of candidate models
# ---------------------------------------------------------------------------
results = {}
 
# --- Linear Regression (on Lasso-selected features) ---
lr = LinearRegression()
lr.fit(X_train_lasso, y_train)
lr_pred = lr.predict(X_test_lasso)
lr_mse = mean_squared_error(y_test, lr_pred)
results["LinearRegression"] = (lr, lr_mse)
print(f"LinearRegression MSE: {lr_mse:.4f}")
 
# --- Random Forest with hyperparameter search ---
param_dist = {
    "n_estimators": np.arange(50, 300, 25),
    "max_depth": np.arange(2, 20, 2),
    "min_samples_leaf": np.arange(1, 10),
    "min_samples_split": np.arange(2, 10),
}
 
rf = RandomForestRegressor(random_state=9)
 
rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="neg_mean_squared_error",
    random_state=9,
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)
 
best_rf = rf_search.best_estimator_
rf_pred = best_rf.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_pred)
results["RandomForestRegressor"] = (best_rf, rf_mse)
print(f"RandomForestRegressor MSE: {rf_mse:.4f} (best params: {rf_search.best_params_})")


# 8. Pick the best model (lowest test MSE)
# ---------------------------------------------------------------------------
best_model_name = min(results, key=lambda name: results[name][1])
best_model, best_mse = results[best_model_name]
 
print(f"\nRecommended model: {best_model_name}")
print(f"best_mse on test set: {best_mse:.4f}")
 
if best_mse < 3:
    print("Target achieved: best_mse < 3")
else:
    print("Target NOT met (best_mse >= 3) — consider more hyperparameter "
          "search iterations, additional features, or gradient boosting models.")



Features used in X: ['amount', 'release_year', 'rental_rate', 'length', 'replacement_cost', 'NC-17', 'PG', 'PG-13', 'R', 'amount_2', 'length_2', 'rental_rate_2', 'deleted_scenes', 'behind_the_scenes']
Lasso-selected features: ['amount', 'amount_2', 'length_2', 'rental_rate_2']
LinearRegression MSE: 3.0924
RandomForestRegressor MSE: 2.0614 (best params: {'n_estimators': 150, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_depth': 14})

Recommended model: RandomForestRegressor
best_mse on test set: 2.0614
Target achieved: best_mse < 3
